# 05 — PPO Training

Trains Stable-Baselines3 `PPO("MlpPolicy", env)` on `InventoryEnv` using `agents/ppo/agent.py`.

It uses the same evaluation schedule and environment settings as the DQN run in `04_dqn_training.ipynb`, so the two training curves can be compared directly.

- **EvalCallback** evaluates on a separate environment and saves the best weights to `models/ppo/best_model.zip`.
- **CheckpointCallback** saves periodic weights to `models/ppo/checkpoints/`.
- In Colab, the `.zip` files are also copied to `MyDrive/inventory-rl/models/ppo/`.

In Colab, the project folder must already be at `/content` (for example by cloning the repo there).

In [ ]:
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "gymnasium==1.4.0", "stable-baselines3==2.9.2"],
        check=True,
    )
except ImportError:
    IN_COLAB = False

# Make the project packages importable when the notebook runs from notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from agents.ppo.agent import MODELS_DIR, train_ppo

print("In Colab:", IN_COLAB)
print("Project root:", PROJECT_ROOT)
print("Models dir:", MODELS_DIR)

## 1. Train

`config/environment_config.yaml` and `config/experiment_config.yaml` are currently empty, so this run uses the `InventoryEnv` defaults and the PPO defaults in `agents/ppo/agent.py`. Keep `total_timesteps`, `eval_freq`, and `n_eval_episodes` the same as the DQN notebook for a fair comparison.

In [ ]:
model, best_path, final_path = train_ppo(
    total_timesteps=200_000,
    env_kwargs={},
    seed=42,
    eval_freq=5_000,
    n_eval_episodes=20,
    checkpoint_freq=20_000,
)

print("Best model:", best_path)
print("Final model:", final_path)

## 2. Evaluation history

Mean evaluation reward at each `EvalCallback` evaluation (higher is better; rewards are negative costs).

In [ ]:
import numpy as np

log = np.load(MODELS_DIR / "logs" / "evaluations.npz")
for timesteps, rewards in zip(log["timesteps"], log["results"]):
    print(f"{int(timesteps):>8}  mean eval reward {rewards.mean():10.2f}")